In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html
from dash.dependencies import Input, Output

In [ ]:
client = bigquery.Client.from_service_account_json("C:/Users/NMTRAN/.dbt/keyfile.json")
dataset = "USER_ACCQUISITION"


In [ ]:
# Load Fact Table and Dimension Tables
query = f"""
SELECT 
    f.total_revenue,
    f.total_users,
    f.total_spend,
    f.arpu,
    f.roas,
    f.conversion_rate,
    f.forecasted_revenue,
    f.high_roas_prob,
    p.platform,
    c.campaign_name,
    n.network_name
FROM `{client.project}.{dataset}.fact_campaign_performance` f
JOIN `{client.project}.{dataset}.dim_campaign` c
    ON f.campaign_id = c.campaign_id
JOIN `{client.project}.{dataset}.dim_network` n
    ON f.network_id = n.network_id
JOIN `{client.project}.{dataset}.dim_platform` p
    ON f.platform_id = p.platform_id
"""
df = client.query(query).to_dataframe()

In [ ]:
# Identify top campaigns per platform
top_campaigns = df.groupby("platform").apply(lambda x: x.loc[x.roas.idxmax()]).reset_index(drop=True)

In [ ]:
# Dash App
app = Dash(__name__)
app.title = "Voodoo UA Campaign Dashboard"

app.layout = html.Div([
    html.H1("Voodoo User Acquisition Dashboard", style={'textAlign': 'center'}),
    
    html.Div([
        html.Label("Select Platform:", style={'fontWeight': 'bold'}),
        dcc.Dropdown(
            options=[{"label": p, "value": p} for p in df['platform'].unique()],
            value="android",
            id="platform-dropdown"
        )
    ], style={'width': '50%', 'margin': 'auto'}),
    
    html.Br(),
    
    html.Div([
        html.H2("Top Campaign Metrics", style={'textAlign': 'center'}),
        html.Div(id="top-campaign-summary", style={'textAlign': 'center', 'fontSize': 16, 'marginBottom': 20})
    ]),
    
    dcc.Graph(id="roas-bar"),
    dcc.Graph(id="forecast-line"),
    dcc.Graph(id="high-roas-prob")
])

# Callbacks for interactive visualizations
@app.callback(
    Output("roas-bar", "figure"),
    Output("forecast-line", "figure"),
    Output("high-roas-prob", "figure"),
    Output("top-campaign-summary", "children"),
    Input("platform-dropdown", "value")
)
def update_dashboard(selected_platform):
    df_platform = df[df.platform == selected_platform]
    
    # Top campaign info
    top = df_platform.loc[df_platform.roas.idxmax()]
    summary_text = f"Top Campaign: {top.campaign_name} | ROAS: {top.roas:.2f} | ARPU: {top.arpu:.2f} | Forecasted Revenue: ${top.forecasted_revenue:,.2f}"
    
    # ROAS per campaign
    fig_roas = px.bar(
        df_platform, x="campaign_name", y="roas", color="network_name",
        title=f"ROAS per Campaign ({selected_platform})",
        labels={"roas": "ROAS", "campaign_name": "Campaign"}
    )
    
    # Forecasted revenue per campaign
    fig_forecast = px.line(
        df_platform, x="campaign_name", y="forecasted_revenue", color="network_name",
        title=f"Forecasted Revenue per Campaign ({selected_platform})",
        labels={"forecasted_revenue": "Forecasted Revenue ($)", "campaign_name": "Campaign"}
    )
    
    # High ROAS prbabi per campaign
    fig_prob = px.bar(
        df_platform, x="campaign_name", y="high_roas_prob", color="network_name",
        title=f"High ROAS Probability per Campaign ({selected_platform})",
        labels={"high_roas_prob": "High ROAS Probability", "campaign_name": "Campaign"},
        text=df_platform['high_roas_prob'].apply(lambda x: f"{x:.2f}")
    )
    
    fig_prob.update_traces(textposition='outside')
    
    return fig_roas, fig_forecast, fig_prob, summary_text

# Run Dash App
if __name__ == "__main__":
    app.run(debug=True)

C:\Users\NMTRAN\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning:

BigQuery Storage module not found, fetch data with the REST endpoint instead.

C:\Users\NMTRAN\AppData\Local\Temp\ipykernel_19392\3207011479.py:39: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

